# Capítulo 3 · Laboratório 3: o mesmo preço de vários jeitos
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap03.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Tudo é feito do zero, com pandas e matplotlib: ponto e figura, Heikin-Ashi, renko, kagi, three line break, escalas, continuidade temporal e scanners de volume.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap03.pkl"
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
ATIVOS = {"ITUB4.SA": "2024-06-01", "PETR4.SA": "2023-06-01", "BBDC4.SA": "2025-01-01", "VALE3.SA": "2018-01-01",
          "WEGE3.SA": "2018-01-01", "^BVSP": "2000-01-01", "MGLU3.SA": "2014-01-01", "BBAS3.SA": "2025-06-01"}


def baixa(t, ini):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=False, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for t, ini in ATIVOS.items():
    if t not in D:
        D[t] = baixa(t, ini)
confere({t: D[t] for t in ATIVOS})
pd.to_pickle(D, CACHE)

## Parte 1: limpeza de dados

In [ ]:
def limpa(d, indice=False):
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"])
    if not indice:
        ok &= d["Volume"] > 0
    return d[ok].copy()


print("Parte 1. Limpeza de dados: pregões descartados (preço zerado, máxima abaixo da mínima ou volume zero)")
for t in ATIVOS:
    antes = len(D[t])
    D[t] = limpa(D[t], indice=t.startswith("^"))
    print(f"   {t}: {antes - len(D[t])} de {antes}")

## Parte 2: ponto e figura do zero

In [ ]:
def ponto_e_figura(d, caixa, reversao=3):
    """Colunas [tipo, caixa inicial, caixa final, data de início], pela máxima e pela mínima do dia."""
    cols = []
    for data, (h, l) in zip(d.index, d[["High", "Low"]].to_numpy()):
        alto, baixo = int(np.floor(h / caixa)), int(np.ceil(l / caixa))
        if not cols:
            cols.append(["X", baixo, alto, data])
            continue
        tipo, ini, fim, _ = cols[-1]
        if tipo == "X":
            if alto > fim:
                cols[-1][2] = alto
            elif fim - baixo >= reversao:
                cols.append(["O", fim - 1, baixo, data])
        else:
            if baixo < fim:
                cols[-1][2] = baixo
            elif alto - fim >= reversao:
                cols.append(["X", fim + 1, alto, data])
    return cols


def sinais(cols):
    """Sinal de compra: X acima do topo da coluna de X anterior; venda: O abaixo do fundo da coluna de O anterior."""
    s = []
    for i in range(2, len(cols)):
        if cols[i][0] == "X" and cols[i][2] > cols[i - 2][2]:
            s.append((i, "compra"))
        if cols[i][0] == "O" and cols[i][2] < cols[i - 2][2]:
            s.append((i, "venda"))
    return s


CAIXA_IB = 2000
PF = ponto_e_figura(D["^BVSP"].loc["2020-01-01":], CAIXA_IB)
SINAIS_PF = sinais(PF)
print(f"\nParte 2. Ponto e figura do Ibovespa desde 2020 (caixa de {nr(CAIXA_IB, 0)} pontos, reversão de 3):"
      f" {len(PF)} colunas; {sum(1 for _, k in SINAIS_PF if k == 'compra')} sinais de compra e"
      f" {sum(1 for _, k in SINAIS_PF if k == 'venda')} de venda; última coluna: {PF[-1][0]}")
topo = max(max(c[1], c[2]) for c in PF[-25:])
base = min(min(c[1], c[2]) for c in PF[-25:])
for nivel in range(topo, base - 1, -1):
    linha = "".join(c[0] if min(c[1], c[2]) <= nivel <= max(c[1], c[2]) else "." for c in PF[-25:])
    print(f"   {nr(nivel * CAIXA_IB, 0):>8s} {linha}")

## Parte 3: Heikin-Ashi

In [ ]:
def heikin_ashi(d):
    ha = pd.DataFrame(index=d.index)
    ha["Close"] = d[["Open", "High", "Low", "Close"]].mean(axis=1)
    ab = [(d["Open"].iloc[0] + d["Close"].iloc[0]) / 2]
    for i in range(1, len(d)):
        ab.append((ab[-1] + ha["Close"].iloc[i - 1]) / 2)
    ha["Open"] = ab
    ha["High"] = pd.concat([d["High"], ha["Open"], ha["Close"]], axis=1).max(axis=1)
    ha["Low"] = pd.concat([d["Low"], ha["Open"], ha["Close"]], axis=1).min(axis=1)
    return ha[["Open", "High", "Low", "Close"]]


p6 = D["PETR4.SA"].loc["2026-04-01":]
HA = heikin_ashi(p6)
seq = lambda s: (s != s.shift()).cumsum().value_counts().mean()
print(f"\nParte 3. Heikin-Ashi da PETR4 (abr. a set./2026): sequência média de velas da mesma cor"
      f" {nr(seq(p6['Close'] >= p6['Open']), 1)} no candle comum e {nr(seq(HA['Close'] >= HA['Open']), 1)} no Heikin-Ashi;"
      f" diferença média entre o fechamento HA e o real: R$ {nr((HA['Close'] - p6['Close']).abs().mean())}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def candles(ax, d, w=0.6):
    """Padrão do livro: candle branco = alta, preto = baixa."""
    for i, (o, h, l, c) in enumerate(d[["Open", "High", "Low", "Close"]].to_numpy()):
        ax.plot([i, i], [l, h], color="black", lw=0.6)
        ax.add_patch(Rectangle((i - w / 2, min(o, c)), w, max(abs(c - o), 1e-9),
                               facecolor="white" if c >= o else "black", edgecolor="black", lw=0.6))
    ax.set_xlim(-1, len(d))

fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
candles(axs[0], p6); axs[0].set_title("PETR4: velas comuns")
candles(axs[1], HA); axs[1].set_title("PETR4: Heikin-Ashi")
plt.tight_layout(); plt.show()

## Parte 4: renko a fundo

In [ ]:
def atr(d, n=14):
    tr = pd.concat([d["High"] - d["Low"], (d["High"] - d["Close"].shift()).abs(),
                    (d["Low"] - d["Close"].shift()).abs()], axis=1).max(axis=1)
    return tr.rolling(n).mean()


def renko(fech, tij):
    """Tijolos pelos fechamentos: mesma direção = 1 tijolo além do último; virada = 2 tijolos."""
    out, base, dire = [], fech.iloc[0], 0
    for data, p in fech.items():
        while True:
            if dire >= 0 and p >= base + tij:
                out.append((data, base, base + tij, 1)); base += tij; dire = 1
            elif dire <= 0 and p <= base - tij:
                out.append((data, base, base - tij, -1)); base -= tij; dire = -1
            elif dire == 1 and p <= base - 2 * tij:
                out.append((data, base - tij, base - 2 * tij, -1)); base -= 2 * tij; dire = -1
            elif dire == -1 and p >= base + 2 * tij:
                out.append((data, base + tij, base + 2 * tij, 1)); base += 2 * tij; dire = 1
            else:
                break
    return pd.DataFrame(out, columns=["data", "abre", "fecha", "dir"])


pr = D["PETR4.SA"].loc["2024-10-01":]
TIJ = round(float(atr(pr).dropna().iloc[0]), 2)
RK = renko(pr["Close"], TIJ)
viradas = RK[RK["dir"] != RK["dir"].shift()].iloc[1:]
dif = (pr.loc[viradas["data"], "Close"].values - viradas["fecha"].values) * viradas["dir"].values
RENKO = {"tijolo": TIJ, "pregoes": len(pr), "tijolos": len(RK), "sem": len(pr) - RK["data"].nunique(),
         "mais_de_um": int((RK.groupby("data").size() > 1).sum()), "viradas": len(viradas), "dif": float(dif.mean())}
print(f"\nParte 4. Renko da PETR4 (out./2024 a set./2026), tijolo de R$ {nr(TIJ)} (ATR de 14 dias no início):"
      f" {RENKO['tijolos']} tijolos em {RENKO['pregoes']} pregões; {RENKO['sem']} pregões sem tijolo;"
      f" {RENKO['mais_de_um']} com mais de um; {RENKO['viradas']} viradas de cor; o preço real saiu em média"
      f" R$ {nr(abs(RENKO['dif']))} {'pior' if RENKO['dif'] > 0 else 'melhor'} que o preço do tijolo"
      f" ({pc(abs(RENKO['dif']) / TIJ, 0)} de um tijolo)")
# tijolo fixo de R$ 1,00 para o gráfico "clássico"
RK1 = renko(pr["Close"], 1.0)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for i, r in RK.reset_index(drop=True).iterrows():
    lo, hi = min(r["abre"], r["fecha"]), max(r["abre"], r["fecha"])
    ax.add_patch(Rectangle((i, lo), 0.9, hi - lo, fc="white" if r["dir"] > 0 else "black", ec="black", lw=0.5))
ax.set_xlim(-1, len(RK) + 1); ax.set_ylim(RK[["abre", "fecha"]].min().min() * 0.98, RK[["abre", "fecha"]].max().max() * 1.02)
ax.set_title(f"PETR4: renko com tijolo de R$ {TIJ}"); plt.show()

## Parte 5: kagi e three line break

In [ ]:
def kagi(fech, rev):
    """Linhas do kagi: muda de direção quando o preço volta 'rev' (fração) contra o último extremo."""
    pts, dire, ext = [(0, fech.iloc[0])], 0, fech.iloc[0]
    for p in fech.iloc[1:]:
        if dire >= 0 and p > ext:
            ext, dire = p, 1
        elif dire <= 0 and p < ext:
            ext, dire = p, -1
        elif dire == 1 and p <= ext * (1 - rev):
            pts.append((len(pts), ext)); ext, dire = p, -1
        elif dire == -1 and p >= ext * (1 + rev):
            pts.append((len(pts), ext)); ext, dire = p, 1
    pts.append((len(pts), ext))
    return [v for _, v in pts]


def three_line_break(fech, n=3):
    linhas = []                                     # (abre, fecha)
    for p in fech:
        if not linhas:
            linhas.append((p, p)); continue
        ab, fe = linhas[-1]
        alta = fe >= ab
        ult = linhas[-n:]
        hi = max(max(a, f) for a, f in ult); lo = min(min(a, f) for a, f in ult)
        mesma = all((f >= a) == alta for a, f in ult) and len(ult) == n
        if alta:
            if p > fe:
                linhas.append((fe, p))
            elif (p < lo if mesma else p < ab):
                linhas.append((ab, p))
        else:
            if p < fe:
                linhas.append((fe, p))
            elif (p > hi if mesma else p > ab):
                linhas.append((ab, p))
    return linhas[1:]


ib2 = D["^BVSP"].loc["2024-10-01":, "Close"]
KAGI = kagi(ib2, 0.04)
TLB = three_line_break(ib2)
print(f"\nParte 5. Ibovespa (out./2024 a set./2026): kagi com reversão de 4% = {len(KAGI) - 1} linhas;"
      f" three line break = {len(TLB)} linhas ({sum(1 for a, f in TLB if f > a)} de alta)")

## Parte 6: escalas

In [ ]:
ib = D["^BVSP"]["Close"]
mg = D["MGLU3.SA"]["Close"]
print(f"\nParte 6. Escalas: Ibovespa de {nr(ib.iloc[0], 0)} (jan./2000) a {nr(ib.iloc[-1], 0)} pontos;"
      f" MGLU3 (preço ajustado) de R$ {nr(mg.loc[:'2016-12-31'].min())} em 2016 a R$ {nr(mg.max())} no topo de"
      f" {mg.idxmax():%m/%Y} ({nr(mg.max() / mg.loc[:'2016-12-31'].min(), 0)} vezes) e R$ {nr(mg.iloc[-1])} em 30/09/2026")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(mg, color="black"); axs[0].set_title("MGLU3: escala aritmética")
axs[1].plot(mg, color="black"); axs[1].set_yscale("log"); axs[1].set_title("MGLU3: escala logarítmica")
plt.tight_layout(); plt.show()

## Parte 7: continuidade temporal

In [ ]:
def continuidade(d):
    ult = d["Close"].iloc[-1]
    ab = {"dia": d["Open"].iloc[-1], "semana": d["Open"].resample("W-FRI").first().iloc[-1],
          "mês": d["Open"].resample("ME").first().iloc[-1]}
    acima = [ult > a for a in ab.values()]
    return ("compradores no controle" if all(acima) else "vendedores no controle" if not any(acima) else "misto"), ab


print("\nParte 7. Continuidade temporal em 30/09/2026 (fechamento contra a abertura do dia, da semana e do mês)")
CONT = {}
for t in ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBAS3.SA", "WEGE3.SA"]:
    CONT[t] = continuidade(D[t])
    print(f"   {t[:-3]}: {CONT[t][0]}")

## Parte 8: scanner de volume

In [ ]:
v = D["VALE3.SA"].loc["2019-01-01":]
vm = v["Volume"].rolling(20).mean().shift(1)
topo60 = v["High"] >= v["High"].rolling(60).max()                       # máxima de 60 pregões no dia
fecha_baixo = (v["Close"] - v["Low"]) / (v["High"] - v["Low"]) < 0.3     # fecha no terço de baixo da barra
clim = v[(v["Volume"] > 1.6 * vm) & topo60 & fecha_baixo]
clim = clim.assign(depois=(v["Close"].shift(-20) / v["Close"] - 1).reindex(clim.index)).dropna()
CLIMAX = clim.sort_values("depois").index[0]
print(f"\nParte 8. Clímax de compra em VALE3 desde 2019 (máxima de 60 pregões, volume > 1,6x a média de 20,"
      f" fechamento no terço de baixo): {len(clim)} casos; em {pc((clim['depois'] < 0).mean(), 0)} deles o preço"
      f" estava mais baixo 20 pregões depois; exemplo do livro: {CLIMAX:%d/%m/%Y} ({pc(clim.loc[CLIMAX, 'depois'])})")
w = D["WEGE3.SA"]
wm = w["Volume"].rolling(20).mean().shift(1)
caiu = w["Close"].shift(1) / w["Close"].shift(61) - 1
rompe = w["Close"] > w["High"].rolling(20).max().shift(1)
vir = w[(caiu < -0.10) & rompe & (w["Volume"] > 1.5 * wm)]
vir = vir.assign(depois=(w["Close"].shift(-40) / w["Close"] - 1).reindex(vir.index)).dropna()
vir = vir[~vir.index.to_series().diff().dt.days.lt(30).fillna(False)]       # um caso por episódio
VIRADA = vir.sort_values("depois").index[-1]
print(f"   Virada com volume em WEGE3 desde 2018 (queda > 10% em 60 pregões, rompe a máxima de 20 com volume > 1,5x):"
      f" {len(vir)} casos; em {pc((vir['depois'] > 0).mean(), 0)} deles o preço estava mais alto 40 pregões depois;"
      f" exemplo do livro: {VIRADA:%d/%m/%Y} ({pc(vir.loc[VIRADA, 'depois'])})")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `renko` da Parte 4. Mostre onde nasce um tijolo novo, um tamanho inteiro além do último, e onde a virada de cor exige dois tijolos (`2 * tij`)." Confira com a própria Parte 4, que imprime, para a PETR4, tijolo de R$ 0,69, 140 tijolos em 498 pregões, 32 viradas de cor e o preço real R$ 0,35 pior que o preço do tijolo, como no livro. Esse resultado é de teste passado: regra tirada daqui precisa do teste fora da amostra (Capítulo 27).
2. **Adaptar:** "Acrescente [ATIVO] ao dicionário `ATIVOS` da célula de abertura, com o ticker do Yahoo Finance terminado em .SA e a data de início, e inclua o mesmo ticker na lista da Parte 7. Não mude a regra da continuidade temporal." Antes de olhar o ativo novo, confira se a Parte 7 continua com ITUB4 e BBAS3 com compradores no controle, VALE3 e WEGE3 com vendedores no controle e PETR4 misto, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, troque a caixa do ponto e figura para 1.000 e para 4.000 pontos. Quantas colunas sobram?
2. Na Parte 4, use um tijolo fixo de R$ 0,50 e de R$ 2,00. O que acontece com o número de viradas de cor?
3. Na Parte 7, ponha as suas ações na lista e rode no dia de hoje.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.